In [2]:
%pip install pyodbc nltk pandas

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.6 MB 645.7 kB/s eta 0:00:02
   ------------- -------------------------- 0.5/1.6 MB 645.7 kB/s eta 0:00:02
   ------------- -------------------------- 0.5/1.6 MB 645.7 kB/s eta 0:00:02
   -------------------- ------------------- 0.8/1.6 MB 578.7 kB/s eta 0:00:02
   -------------------- ------------------- 0.8/1.6 MB 578.7 kB/s eta 0:00:02
   --------------------------- ------------ 1.0/1.6 MB 585.6 kB/s eta 0:00:01
   --------------------------- ------------ 1.0/1.6 MB 585.6 kB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.6 MB 599.2 kB/s eta 0:00:01
   -------------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# pip install pandas nltk pyodbc sqlalchemy

import pandas as pd
import pyodbc
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [9]:

print(pyodbc.drivers())

['SQL Server', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [16]:

import os
# Initialize sentiment analyzer
sia = SentimentIntensityAnalyzer()

# Function to fetch data from SQL Server
def fetch_data_from_sql():
    conn_str = (
        "Driver={ODBC Driver 17 for SQL Server};"
        "Server=localhost;"  # your local server
        "Database=PortfolioProject_MarketingAnalytics;"  # your database
        "Trusted_Connection=yes;"  # Windows Authentication
    )
    
    # Use context manager to auto-close connection
    try:
        with pyodbc.connect(conn_str, timeout=5) as conn:
            print("Connection successful!")
            query = "SELECT ReviewID, CustomerID, ProductID, ReviewDate, Rating, ReviewText FROM customer_reviews"
            df = pd.read_sql(query, conn)
            return df
    except Exception as e:
        print("Connection failed:", e)
        return None

# Fetch the data
customer_reviews_df = fetch_data_from_sql()

# Make sure data was fetched
if customer_reviews_df is not None and not customer_reviews_df.empty:
    # Sentiment analysis
    def calculate_sentiment(review):
        return sia.polarity_scores(review)['compound']

    def categorize_sentiment(score, rating):
        if score > 0.05:
            if rating >= 4:
                return 'Positive'
            elif rating == 3:
                return 'Mixed Positive'
            else:
                return 'Mixed Negative'
        elif score < -0.05:
            if rating <= 2:
                return 'Negative'
            elif rating == 3:
                return 'Mixed Negative'
            else:
                return 'Mixed Positive'
        else:
            if rating >= 4:
                return 'Positive'
            elif rating <= 2:
                return 'Negative'
            else:
                return 'Neutral'

    def sentiment_bucket(score):
        if score >= 0.5:
            return '0.5 to 1.0'
        elif 0.0 <= score < 0.5:
            return '0.0 to 0.49'
        elif -0.5 <= score < 0.0:
            return '-0.49 to 0.0'
        else:
            return '-1.0 to -0.5'

    # Apply sentiment analysis
    customer_reviews_df['SentimentScore'] = customer_reviews_df['ReviewText'].apply(calculate_sentiment)
    customer_reviews_df['SentimentCategory'] = customer_reviews_df.apply(
        lambda row: categorize_sentiment(row['SentimentScore'], row['Rating']), axis=1)
    customer_reviews_df['SentimentBucket'] = customer_reviews_df['SentimentScore'].apply(sentiment_bucket)

    # Display sample
    print(customer_reviews_df.head())

    # Save to CSV
     # Set Downloads folder path
    downloads_folder = os.path.join(os.path.expanduser("~"), "Downloads")
    os.makedirs(downloads_folder, exist_ok=True)  # ensures folder exists
    file_name = "fact_customer_reviews_with_sentiment.csv"
    file_path = os.path.join(downloads_folder, file_name)

    # Save CSV
    customer_reviews_df.to_csv(file_path, index=False)
    print(f"CSV successfully saved at: {file_path}")
else:
    print("No data fetched from SQL Server. Check database/table.")

Connection successful!
   ReviewID  CustomerID  ProductID  ReviewDate  Rating  \
0         1          77         18  2023-12-23       3   
1         2          80         19  2024-12-25       5   
2         3          50         13  2025-01-26       4   
3         4          78         15  2025-04-21       3   
4         5          64          2  2023-07-16       3   

                                 ReviewText  SentimentScore SentimentCategory  \
0   Average  experience,  nothing  special.         -0.3089    Mixed Negative   
1            The  quality  is    top-notch.          0.0000          Positive   
2   Five  stars  for  the  quick  delivery.          0.0000          Positive   
3  Good  quality,  but  could  be  cheaper.          0.2382    Mixed Positive   
4   Average  experience,  nothing  special.         -0.3089    Mixed Negative   

  SentimentBucket  
0    -0.49 to 0.0  
1     0.0 to 0.49  
2     0.0 to 0.49  
3     0.0 to 0.49  
4    -0.49 to 0.0  
CSV successfully save

C:\Users\hamza\AppData\Local\Temp\ipykernel_4836\2262202394.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
